In [44]:
import os
from pathlib import Path

import polars as pl
from sklearn.model_selection import train_test_split

In [45]:
DATASET_PATH = Path("datasets/dataset_20260707.csv")

In [46]:
DATABASE_URI = os.environ["DATABASE_URI"]

In [47]:
RANDOM_SEED = 42

# Chargement des données

In [48]:
df_dataset = pl.read_csv(DATASET_PATH, schema_overrides={"cluster_id": pl.String})

In [49]:
df_dataset

identifiant_unique_i,identifiant_unique_j,label,cluster_id
str,str,bool,str
"""07e07779-388e-5971-947b-b999ae…","""lokki_67692fc641d8badbf051ec32""",true,"""7758461304653219539"""
"""085352c0-77c6-599d-b54a-02ef2a…","""772950c8-5202-5394-b0ae-8ebbda…",false,null
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5487990001""",false,null
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488009001""",false,null
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488015001""",false,null
…,…,…,…
"""refashion_TLC-REFASHION-PAV-32…","""refashion_TLC-REFASHION-PAV-34…",true,"""9a14e7d6-f708-58fd-bcbb-3d3c2c…"
"""refashion_TLC-REFASHION-PAV-33…","""refashion_TLC-REFASHION-PAV-34…",true,"""1f8fb158-8b9a-54b5-81ca-d9d542…"
"""refashion_TLC-REFASHION-PAV-33…","""refashion_TLC-REFASHION-PAV-34…",true,"""05f37598-e1d8-40a1-bb9b-5be142…"


In [50]:
df_dataset.group_by("label").len()

label,len
bool,u32
true,1129
false,1000


# Chargement des variables d'intérêt depuis la base

In [22]:
df_dataset.write_database(
    "luis.dataset_tmp", connection=DATABASE_URI, if_table_exists="replace"
)

129

In [23]:
sql_dataset_features = """
with actions_by_actor as (
-- D'abord on calcule les actions des parents sur la base de leurs enfants
select 
	qv.identifiant_unique as acteur_id,
	(1 = any(array_agg(qv2.action_id))) as action_reparer,
	(2 = any(array_agg(qv2.action_id))) as action_acheter,
	(3 = any(array_agg(qv2.action_id))) as action_revendre,
	(4 = any(array_agg(qv2.action_id))) as action_donner,
	(5 = any(array_agg(qv2.action_id))) as action_louer,
	(6 = any(array_agg(qv2.action_id))) as action_mettreenlocation,
	(7 = any(array_agg(qv2.action_id))) as action_emprunter,
	(8 = any(array_agg(qv2.action_id))) as action_preter,
	(9 = any(array_agg(qv2.action_id))) as action_echanger,
	(11 = any(array_agg(qv2.action_id))) as action_trier,
	(12 = any(array_agg(qv2.action_id))) as action_rapporter
from
	qfdmo_vueacteur qv
inner join qfdmo_vueacteur qv3 on
	qv.est_parent
	and qv.identifiant_unique = qv3.parent_id
	and qv3.statut <> 'SUPPRIME'
inner join qfdmo_vuepropositionservice qv2 on
	qv3.identifiant_unique = qv2.acteur_id
group by
	1
union all -- on joint les actions des enfants
select
	acteur_id,
	(1 = any(array_agg(action_id))) as action_reparer,
	(2 = any(array_agg(action_id))) as action_acheter,
	(3 = any(array_agg(action_id))) as action_revendre,
	(4 = any(array_agg(action_id))) as action_donner,
	(5 = any(array_agg(action_id))) as action_louer,
	(6 = any(array_agg(action_id))) as action_mettreenlocation,
	(7 = any(array_agg(action_id))) as action_emprunter,
	(8 = any(array_agg(action_id))) as action_preter,
	(9 = any(array_agg(action_id))) as action_echanger,
	(11 = any(array_agg(action_id))) as action_trier,
	(12 = any(array_agg(action_id))) as action_rapporter
from
	qfdmo_vuepropositionservice
group by
	acteur_id
),
-- Sélection des variables à la maille acteur avec les actions précédemment sélectionnées
features as (
select
	identifiant_unique,
	nom,
	description,
	acteur_type_id,
	adresse,
	adresse_complement,
	code_postal,
	ville,
	url,
	email,
	"location",
	telephone,
	nom_commercial,
	nom_officiel,
	siren,
	siret,
	source_id,
	naf_principal,
	horaires_osm,
	horaires_description,
	public_accueilli,
	reprise,
	exclusivite_de_reprisereparation,
	uniquement_sur_rdv,
	consignes_dacces,
	action_principale_id,
	lieu_prestation,
	latitude,
	longitude,
	code_commune_insee,
	epci_id,
	aba.*
from
	qfdmo_vueacteur qv
left join actions_by_actor aba on
	qv.identifiant_unique = aba.acteur_id)
-- Join avec les acteurs sélectionnés dans le dataset
select 
	dt.*,
	f.nom as nom_i,
	f.description as description_i,
	f.acteur_type_id as acteur_type_id_i,
	f.adresse as adresse_i,
	f.adresse_complement as adresse_complement_i,
	f.code_postal as code_postal_i,
	f.ville as ville_i,
	f.url as url_i,
	f.email as email_i,
	f.telephone as telephone_i,
	f.nom_commercial as nom_commercial_i,
	f.nom_officiel as nom_officiel_i,
	f.siren as siren_i,
	f.siret as siret_i,
	f.source_id as source_id_i,
	f.naf_principal as naf_principal_i,
	f.horaires_osm as horaires_osm_i,
	f.horaires_description as horaires_description_i,
	f.public_accueilli as public_accueilli_i,
	f.reprise as reprise_i,
	f.exclusivite_de_reprisereparation as exclusivite_de_reprisereparation_i,
	f.uniquement_sur_rdv as uniquement_sur_rdv_i,
	f.consignes_dacces as consignes_dacces_i,
	f.action_principale_id as action_principale_id_i,
	f.lieu_prestation as lieu_prestation_i,
	f.latitude as latitude_i,
	f.longitude as longitude_i,
	f.code_commune_insee as code_commune_insee_i,
	f.epci_id as epci_id_i,
	f.action_reparer as action_reparer_i,
	f.action_acheter as action_acheter_i,
	f.action_revendre as action_revendre_i,
	f.action_donner as action_donner_i,
	f.action_louer as action_louer_i,
	f.action_mettreenlocation as action_mettreenlocation_i,
	f.action_emprunter as action_emprunter_i,
	f.action_preter as action_preter_i,
	f.action_echanger as action_echanger_i,
	f.action_trier as action_trier_i,
	f.action_rapporter as action_rapporter_i,
	f2.nom as nom_j,
	f2.description as description_j,
	f2.acteur_type_id as acteur_type_id_j,
	f2.adresse as adresse_j,
	f2.adresse_complement as adresse_complement_j,
	f2.code_postal as code_postal_j,
	f2.ville as ville_j,
	f2.url as url_j,
	f2.email as email_j,
	f2.telephone as telephone_j,
	f2.nom_commercial as nom_commercial_j,
	f2.nom_officiel as nom_officiel_j,
	f2.siren as siren_j,
	f2.siret as siret_j,
	f2.source_id as source_id_j,
	f2.naf_principal as naf_principal_j,
	f2.horaires_osm as horaires_osm_j,
	f2.horaires_description as horaires_description_j,
	f2.public_accueilli as public_accueilli_j,
	f2.reprise as reprise_j,
	f2.exclusivite_de_reprisereparation as exclusivite_de_reprisereparation_j,
	f2.uniquement_sur_rdv as uniquement_sur_rdv_j,
	f2.consignes_dacces as consignes_dacces_j,
	f2.action_principale_id as action_principale_id_j,
	f2.lieu_prestation as lieu_prestation_j,
	f2.latitude as latitude_j,
	f2.longitude as longitude_j,
	f2.code_commune_insee as code_commune_insee_j,
	f2.epci_id as epci_id_j,
	f2.action_reparer as action_reparer_j,
	f2.action_acheter as action_acheter_j,
	f2.action_revendre as action_revendre_j,
	f2.action_donner as action_donner_j,
	f2.action_louer as action_louer_j,
	f2.action_mettreenlocation as action_mettreenlocation_j,
	f2.action_emprunter as action_emprunter_j,
	f2.action_preter as action_preter_j,
	f2.action_echanger as action_echanger_j,
	f2.action_trier as action_trier_j,
	f2.action_rapporter as action_rapporter_j
from
	luis.dataset_tmp dt
left join features f on
	dt.identifiant_unique_i = f.identifiant_unique
left join features f2 on
	dt.identifiant_unique_j = f2.identifiant_unique
"""

In [24]:
df_features = pl.read_database_uri(sql_dataset_features, uri=DATABASE_URI)

In [51]:
df_features.shape

(2129, 84)

In [52]:
# Assert no duplicates
assert df_features.group_by(
    ["identifiant_unique_i", "identifiant_unique_j"]
).len().select(pl.col("len")).sum().item() == len(df_features)

In [53]:
df_features.describe()

statistic,identifiant_unique_i,identifiant_unique_j,label,cluster_id,nom_i,description_i,acteur_type_id_i,adresse_i,adresse_complement_i,code_postal_i,ville_i,url_i,email_i,telephone_i,nom_commercial_i,nom_officiel_i,siren_i,siret_i,source_id_i,naf_principal_i,horaires_osm_i,horaires_description_i,public_accueilli_i,reprise_i,exclusivite_de_reprisereparation_i,uniquement_sur_rdv_i,consignes_dacces_i,action_principale_id_i,lieu_prestation_i,latitude_i,longitude_i,code_commune_insee_i,epci_id_i,action_reparer_i,action_acheter_i,action_revendre_i,…,adresse_j,adresse_complement_j,code_postal_j,ville_j,url_j,email_j,telephone_j,nom_commercial_j,nom_officiel_j,siren_j,siret_j,source_id_j,naf_principal_j,horaires_osm_j,horaires_description_j,public_accueilli_j,reprise_j,exclusivite_de_reprisereparation_j,uniquement_sur_rdv_j,consignes_dacces_j,action_principale_id_j,lieu_prestation_j,latitude_j,longitude_j,code_commune_insee_j,epci_id_j,action_reparer_j,action_acheter_j,action_revendre_j,action_donner_j,action_louer_j,action_mettreenlocation_j,action_emprunter_j,action_preter_j,action_echanger_j,action_trier_j,action_rapporter_j
str,str,str,f64,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,f64,f64,str,f64,str,f64,f64,str,f64,f64,f64,f64,…,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,f64,f64,str,f64,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""2129""","""2129""",2129.0,"""1129""","""2129""","""2129""",2129.0,"""2129""","""2129""","""2129""","""2129""","""2129""","""2129""","""2129""","""2129""","""2129""","""2129""","""2129""",1664.0,"""2129""","""2129""","""2129""","""2129""","""2129""",392.0,452.0,"""2129""",13.0,"""2115""",2127.0,2127.0,"""2129""",2068.0,2089.0,2089.0,2089.0,…,"""2129""","""2129""","""2129""","""2129""","""2129""","""2129""","""2129""","""2129""","""2129""","""2129""","""2129""",2062.0,"""2129""","""2129""","""2129""","""2129""","""2129""",362.0,343.0,"""2129""",11.0,"""2128""",2125.0,2125.0,"""2129""",2047.0,2118.0,2118.0,2118.0,2118.0,2118.0,2118.0,2118.0,2118.0,2118.0,2118.0,2118.0
"""null_count""","""0""","""0""",0.0,"""1000""","""0""","""0""",0.0,"""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""",465.0,"""0""","""0""","""0""","""0""","""0""",1737.0,1677.0,"""0""",2116.0,"""14""",2.0,2.0,"""0""",61.0,40.0,40.0,40.0,…,"""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""",67.0,"""0""","""0""","""0""","""0""","""0""",1767.0,1786.0,"""0""",2118.0,"""1""",4.0,4.0,"""0""",82.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0
"""mean""",null,null,0.530296,null,null,null,6.04791,null,null,null,null,null,null,null,null,null,null,null,112.797476,null,null,null,null,null,0.0,0.011062,null,11.0,null,3.008522,45.678555,null,570.644101,0.109143,0.023456,0.01101,…,null,null,null,null,null,null,null,null,null,null,null,124.037827,null,null,null,null,null,0.002762,0.026239,null,10.363636,null,3.021885,45.728445,null,559.532975,0.166195,0.030689,0.005666,0.068933,0.008026,0.0,0.018886,0.0,0.007082,0.77526,0.009443
"""std""",null,null,null,null,null,null,2.794758,null,null,null,null,null,null,null,null,null,null,null,104.98351,null,null,null,null,null,null,null,null,0.0,null,5.88377,5.193348,null,367.047333,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,107.993641,null,null,null,null,null,null,null,null,2.110579,null,5.721949,4.957962,null,361.92289,null,null,null,null,null,null,null,null,null,null,null
"""min""","""00a193b3-b145-5f06-aa29-515d1f…","""1e0bfede-8c6b-5469-9b92-b458bf…",0.0,"""00a193b3-b145-5f06-aa29-515d1f…","""1000 MAINS ET MERVEILLES""","""""",1.0,"""""","""""","""""","""""","""""","""""","""""","""""","""""","""""","""""",1.0,"""""","""""","""""","""""","""""",0.0,0.0,"""""",11.0,"""""",-61.632734,-21.3806,"""""",8.0,0.0,0.0,0.0,…,"""""","""""","""""","""""","""""","""""","""""","""""","""""","""""","""""",1.0,"""""","

# Traitement des "__empty__"

In [54]:
df_features_preprocessed = df_features.with_columns(
    pl.selectors.by_dtype(pl.String)
    .exclude("cluster_id")
    .str.strip_chars()
    .replace("__empty__", None)
    .replace("", None)
)

In [55]:
df_features_preprocessed.describe()

statistic,identifiant_unique_i,identifiant_unique_j,label,cluster_id,nom_i,description_i,acteur_type_id_i,adresse_i,adresse_complement_i,code_postal_i,ville_i,url_i,email_i,telephone_i,nom_commercial_i,nom_officiel_i,siren_i,siret_i,source_id_i,naf_principal_i,horaires_osm_i,horaires_description_i,public_accueilli_i,reprise_i,exclusivite_de_reprisereparation_i,uniquement_sur_rdv_i,consignes_dacces_i,action_principale_id_i,lieu_prestation_i,latitude_i,longitude_i,code_commune_insee_i,epci_id_i,action_reparer_i,action_acheter_i,action_revendre_i,…,adresse_j,adresse_complement_j,code_postal_j,ville_j,url_j,email_j,telephone_j,nom_commercial_j,nom_officiel_j,siren_j,siret_j,source_id_j,naf_principal_j,horaires_osm_j,horaires_description_j,public_accueilli_j,reprise_j,exclusivite_de_reprisereparation_j,uniquement_sur_rdv_j,consignes_dacces_j,action_principale_id_j,lieu_prestation_j,latitude_j,longitude_j,code_commune_insee_j,epci_id_j,action_reparer_j,action_acheter_j,action_revendre_j,action_donner_j,action_louer_j,action_mettreenlocation_j,action_emprunter_j,action_preter_j,action_echanger_j,action_trier_j,action_rapporter_j
str,str,str,f64,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,f64,f64,str,f64,str,f64,f64,str,f64,f64,f64,f64,…,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,f64,f64,str,f64,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""2129""","""2129""",2129.0,"""1129""","""2129""","""95""",2129.0,"""2097""","""214""","""2105""","""2118""","""338""","""175""","""604""","""670""","""14""","""1079""","""948""",1664.0,"""156""","""7""","""53""","""1655""","""389""",392.0,452.0,"""319""",13.0,"""558""",2127.0,2127.0,"""2068""",2068.0,2089.0,2089.0,2089.0,…,"""2070""","""149""","""2109""","""2108""","""257""","""219""","""534""","""550""","""8""","""956""","""939""",2062.0,"""270""","""5""","""62""","""1547""","""259""",362.0,343.0,"""361""",11.0,"""423""",2125.0,2125.0,"""2047""",2047.0,2118.0,2118.0,2118.0,2118.0,2118.0,2118.0,2118.0,2118.0,2118.0,2118.0,2118.0
"""null_count""","""0""","""0""",0.0,"""1000""","""0""","""2034""",0.0,"""32""","""1915""","""24""","""11""","""1791""","""1954""","""1525""","""1459""","""2115""","""1050""","""1181""",465.0,"""1973""","""2122""","""2076""","""474""","""1740""",1737.0,1677.0,"""1810""",2116.0,"""1571""",2.0,2.0,"""61""",61.0,40.0,40.0,40.0,…,"""59""","""1980""","""20""","""21""","""1872""","""1910""","""1595""","""1579""","""2121""","""1173""","""1190""",67.0,"""1859""","""2124""","""2067""","""582""","""1870""",1767.0,1786.0,"""1768""",2118.0,"""1706""",4.0,4.0,"""82""",82.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0
"""mean""",null,null,0.530296,null,null,null,6.04791,null,null,null,null,null,null,null,null,null,null,null,112.797476,null,null,null,null,null,0.0,0.011062,null,11.0,null,3.008522,45.678555,null,570.644101,0.109143,0.023456,0.01101,…,null,null,null,null,null,null,null,null,null,null,null,124.037827,null,null,null,null,null,0.002762,0.026239,null,10.363636,null,3.021885,45.728445,null,559.532975,0.166195,0.030689,0.005666,0.068933,0.008026,0.0,0.018886,0.0,0.007082,0.77526,0.009443
"""std""",null,null,null,null,null,null,2.794758,null,null,null,null,null,null,null,null,null,null,null,104.98351,null,null,null,null,null,null,null,null,0.0,null,5.88377,5.193348,null,367.047333,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,107.993641,null,null,null,null,null,null,null,null,2.110579,null,5.721949,4.957962,null,361.92289,null,null,null,null,null,null,null,null,null,null,null
"""min""","""00a193b3-b145-5f06-aa29-515d1f…","""1e0bfede-8c6b-5469-9b92-b458bf…",0.0,"""00a193b3-b145-5f06-aa29-515d1f…","""1000 MAINS ET MERVEILLES""","""Catégories d'objets proposés :…",1.0,"""/""","""(sortie Carrefour)""","""01000""","""ALLEX""","""http://www.ag2iweb.com""","""Bobimoto@wanadoo.fr""","""+33240963172""","""ALDIS""","""AMBIANCES DU PASSE""","""054800958""","""0

# Traitement de l'absence de latitude/longitude

In [56]:
df_features_preprocessed = df_features_preprocessed.with_columns(
    pl.selectors.contains("latitude", "longitude").fill_null(0)
)

In [57]:
df_features_preprocessed.describe()

statistic,identifiant_unique_i,identifiant_unique_j,label,cluster_id,nom_i,description_i,acteur_type_id_i,adresse_i,adresse_complement_i,code_postal_i,ville_i,url_i,email_i,telephone_i,nom_commercial_i,nom_officiel_i,siren_i,siret_i,source_id_i,naf_principal_i,horaires_osm_i,horaires_description_i,public_accueilli_i,reprise_i,exclusivite_de_reprisereparation_i,uniquement_sur_rdv_i,consignes_dacces_i,action_principale_id_i,lieu_prestation_i,latitude_i,longitude_i,code_commune_insee_i,epci_id_i,action_reparer_i,action_acheter_i,action_revendre_i,…,adresse_j,adresse_complement_j,code_postal_j,ville_j,url_j,email_j,telephone_j,nom_commercial_j,nom_officiel_j,siren_j,siret_j,source_id_j,naf_principal_j,horaires_osm_j,horaires_description_j,public_accueilli_j,reprise_j,exclusivite_de_reprisereparation_j,uniquement_sur_rdv_j,consignes_dacces_j,action_principale_id_j,lieu_prestation_j,latitude_j,longitude_j,code_commune_insee_j,epci_id_j,action_reparer_j,action_acheter_j,action_revendre_j,action_donner_j,action_louer_j,action_mettreenlocation_j,action_emprunter_j,action_preter_j,action_echanger_j,action_trier_j,action_rapporter_j
str,str,str,f64,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,f64,f64,str,f64,str,f64,f64,str,f64,f64,f64,f64,…,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,f64,f64,str,f64,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""2129""","""2129""",2129.0,"""1129""","""2129""","""95""",2129.0,"""2097""","""214""","""2105""","""2118""","""338""","""175""","""604""","""670""","""14""","""1079""","""948""",1664.0,"""156""","""7""","""53""","""1655""","""389""",392.0,452.0,"""319""",13.0,"""558""",2129.0,2129.0,"""2068""",2068.0,2089.0,2089.0,2089.0,…,"""2070""","""149""","""2109""","""2108""","""257""","""219""","""534""","""550""","""8""","""956""","""939""",2062.0,"""270""","""5""","""62""","""1547""","""259""",362.0,343.0,"""361""",11.0,"""423""",2129.0,2129.0,"""2047""",2047.0,2118.0,2118.0,2118.0,2118.0,2118.0,2118.0,2118.0,2118.0,2118.0,2118.0,2118.0
"""null_count""","""0""","""0""",0.0,"""1000""","""0""","""2034""",0.0,"""32""","""1915""","""24""","""11""","""1791""","""1954""","""1525""","""1459""","""2115""","""1050""","""1181""",465.0,"""1973""","""2122""","""2076""","""474""","""1740""",1737.0,1677.0,"""1810""",2116.0,"""1571""",0.0,0.0,"""61""",61.0,40.0,40.0,40.0,…,"""59""","""1980""","""20""","""21""","""1872""","""1910""","""1595""","""1579""","""2121""","""1173""","""1190""",67.0,"""1859""","""2124""","""2067""","""582""","""1870""",1767.0,1786.0,"""1768""",2118.0,"""1706""",0.0,0.0,"""82""",82.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0
"""mean""",null,null,0.530296,null,null,null,6.04791,null,null,null,null,null,null,null,null,null,null,null,112.797476,null,null,null,null,null,0.0,0.011062,null,11.0,null,3.005696,45.635644,null,570.644101,0.109143,0.023456,0.01101,…,null,null,null,null,null,null,null,null,null,null,null,124.037827,null,null,null,null,null,0.002762,0.026239,null,10.363636,null,3.016207,45.64253,null,559.532975,0.166195,0.030689,0.005666,0.068933,0.008026,0.0,0.018886,0.0,0.007082,0.77526,0.009443
"""std""",null,null,null,null,null,null,2.794758,null,null,null,null,null,null,null,null,null,null,null,104.98351,null,null,null,null,null,null,null,null,0.0,null,5.881727,5.376309,null,367.047333,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,107.993641,null,null,null,null,null,null,null,null,2.110579,null,5.718067,5.334643,null,361.92289,null,null,null,null,null,null,null,null,null,null,null
"""min""","""00a193b3-b145-5f06-aa29-515d1f…","""1e0bfede-8c6b-5469-9b92-b458bf…",0.0,"""00a193b3-b145-5f06-aa29-515d1f…","""1000 MAINS ET MERVEILLES""","""Catégories d'objets proposés :…",1.0,"""/""","""(sortie Carrefour)""","""01000""","""ALLEX""","""http://www.ag2iweb.com""","""Bobimoto@wanadoo.fr""","""+33240963172""","""ALDIS""","""AMBIANCES DU PASSE""","""054800958""","""0

# Ajout de cluster_ids pour les négatifs afin de faciliter les splits

In [72]:
df_features_preprocessed = df_features_preprocessed.with_columns(
    pl.coalesce(
        [
            "cluster_id",
            pl.concat_str(
                "identifiant_unique_i", "identifiant_unique_j", separator="___"
            ),
        ]
    ).alias("cluster_id")
)

# Train/test split

In [73]:
train_cluster_ids, test_cluster_ids = train_test_split(
    df_features_preprocessed.select("cluster_id").unique("cluster_id"),
    test_size=0.2,
    random_state=RANDOM_SEED,
)

In [74]:
len(train_cluster_ids), len(test_cluster_ids)

(940, 235)

In [75]:
train_cluster_ids

cluster_id
str
"""ecosystem_EEE-ecosystem-C-1030…"
"""BIB2644.0___mbs_industrie_1988…"
"""BIB13312.0___BIB13321.0"""
"""ecologic_93-DF669-01___ocab_va…"
"""47e7f597-42fe-5292-82eb-716aab…"
…
"""BAL_44.7109809, 1.904558___ali…"
"""aliapur_131015000000005___bout…"
"""772950c8-5202-5394-b0ae-8ebbda…"


In [77]:
df_features_preprocessed_split = df_features_preprocessed.with_columns().with_columns(
    pl.when(
        pl.col("cluster_id").is_in(train_cluster_ids.get_column("cluster_id").to_list())
    )
    .then(pl.lit("train"))
    .when(
        pl.col("cluster_id").is_in(test_cluster_ids.get_column("cluster_id").to_list())
    )
    .then(pl.lit("test"))
    .otherwise(pl.lit("unknown"))
    .alias("split")
)

In [78]:
df_features_preprocessed_split

identifiant_unique_i,identifiant_unique_j,label,cluster_id,nom_i,description_i,acteur_type_id_i,adresse_i,adresse_complement_i,code_postal_i,ville_i,url_i,email_i,telephone_i,nom_commercial_i,nom_officiel_i,siren_i,siret_i,source_id_i,naf_principal_i,horaires_osm_i,horaires_description_i,public_accueilli_i,reprise_i,exclusivite_de_reprisereparation_i,uniquement_sur_rdv_i,consignes_dacces_i,action_principale_id_i,lieu_prestation_i,latitude_i,longitude_i,code_commune_insee_i,epci_id_i,action_reparer_i,action_acheter_i,action_revendre_i,action_donner_i,…,adresse_complement_j,code_postal_j,ville_j,url_j,email_j,telephone_j,nom_commercial_j,nom_officiel_j,siren_j,siret_j,source_id_j,naf_principal_j,horaires_osm_j,horaires_description_j,public_accueilli_j,reprise_j,exclusivite_de_reprisereparation_j,uniquement_sur_rdv_j,consignes_dacces_j,action_principale_id_j,lieu_prestation_j,latitude_j,longitude_j,code_commune_insee_j,epci_id_j,action_reparer_j,action_acheter_j,action_revendre_j,action_donner_j,action_louer_j,action_mettreenlocation_j,action_emprunter_j,action_preter_j,action_echanger_j,action_trier_j,action_rapporter_j,split
str,str,bool,str,str,str,i32,str,str,str,str,str,str,str,str,str,str,str,i32,str,str,str,str,str,bool,bool,str,i32,str,f64,f64,str,i32,bool,bool,bool,bool,…,str,str,str,str,str,str,str,str,str,str,i32,str,str,str,str,str,bool,bool,str,i32,str,f64,f64,str,i32,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,str
"""00a193b3-b145-5f06-aa29-515d1f…","""repair_cafe_poissy_le_champ_de…",true,"""00a193b3-b145-5f06-aa29-515d1f…","""Repair Café Poissy Le Champ de…",null,1,"""80 avenue Fernand Lefebvre""",null,"""78300""","""Poissy""","""https://www.repaircafe.org/caf…",null,null,null,null,null,null,null,null,null,null,null,null,null,true,null,null,"""SUR_PLACE""",2.052674,48.924335,"""78498""",288,true,false,false,false,…,null,"""78300""","""Poissy""",null,null,null,null,null,null,null,5,null,null,null,null,null,null,null,null,null,null,2.051363,48.923819,"""78498""",288,true,false,false,false,false,false,false,false,false,false,false,"""train"""
"""00a193b3-b145-5f06-aa29-515d1f…","""repair_cafe_216""",true,"""00a193b3-b145-5f06-aa29-515d1f…","""Repair Café Poissy Le Champ de…",null,1,"""80 avenue Fernand Lefebvre""",null,"""78300""","""Poissy""","""https://www.repaircafe.org/caf…",null,null,null,null,null,null,null,null,null,null,null,null,null,true,null,null,"""SUR_PLACE""",2.052674,48.924335,"""78498""",288,true,false,false,false,…,null,"""78300""","""Poissy""","""https://www.repaircafe.org/caf…",null,null,null,null,null,null,401,null,null,null,null,null,null,true,null,null,"""SUR_PLACE""",2.052674,48.924335,"""78498""",288,true,false,false,false,false,false,false,false,false,false,false,"""train"""
"""016ab333-09b6-5ee9-a35b-e38c5f…","""citeo_1627076001""",true,"""016ab333-09b6-5ee9-a35b-e38c5f…","""SYNDICAT MIXTE DÉPARTEMENTAL P…",null,10,"""1 Rue du Bois des Dames""",null,"""87290""","""Saint-Sornin-Leulac""",null,null,null,null,null,"""258700251""",null,null,null,null,null,"""Particuliers""",null,null,null,null,null,null,1.261903,46.188935,"""87180""",1215,false,false,false,false,…,null,"""87290""","""Saint-Sornin-Leulac""",null,null,null,null,null,"""258700251""",null,87,null,null,null,"""Particuliers""",null,null,null,null,null,null,1.261903,46.188935,"""87180""",1215,false,false,false,false,false,false,false,false,false,true,false,"""train"""
"""016ab333-09b6-5ee9-a35b-e38c5f…","""citeo_1627076002""",true,"""016ab333-09b6-5ee9-a35b-e38c5f…","""SYNDICAT MIXTE DÉPARTEMENTAL P…",null,10,"""1 Rue du Bois des Dames""",null,"""87290""","""Saint-Sornin-Leulac""",null,null,null,null,null,"""258700251""",null,null,null,null,null,"""Particuliers""",null,null,null,null,null,null,1.261903,46.188935,"""87180""",1215,false,false,false,false,…,null,"""87290""","""Saint-Sornin-Leulac""",null,null,null,null,null,"""258700251""",null,87,null,null,null,"""Particuliers""",null,null,null,null,null,null,1.261903,46.18893

In [79]:
df_features_preprocessed_split.group_by("split").len()

split,len
str,u32
"""test""",399
"""train""",1730


# Sauvegarde

In [80]:
df_features_preprocessed_split.write_parquet(
    DATASET_PATH.parent / f"features_{DATASET_PATH.stem}.parquet"
)